<a href="https://colab.research.google.com/github/keremtayar872-tech/EMU427_Project/blob/main/EMU_427.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import time
import urllib.request


def find_instance_file(filename):
    """Veri dosyasını çalışma dizininde arar; yoksa OR-Library'den otomatik indirir."""
    filepath = os.path.join(os.getcwd(), filename)
    if not os.path.exists(filepath):
        url = f"https://people.brunel.ac.uk/~mastjjb/jeb/orlib/files/{filename}"
        urllib.request.urlretrieve(url, filepath)
    return filepath


def read_airland(filepath):
    """OR-Library airlandX.txt dosyalarını okur ve model parametrelerini döndürür."""
    with open(filepath, "r", encoding="utf-8") as f:
        tokens = f.read().split()

    idx = 0
    N = int(tokens[idx]); idx += 1
    freeze_time = int(tokens[idx]); idx += 1  # Statik problemde kullanılmaz

    E, T, L, g, h = [], [], [], [], []
    S = [[0] * N for _ in range(N)]

    for i in range(N):
        appearance = int(tokens[idx]); idx += 1  # Statik problemde kullanılmaz
        E.append(int(tokens[idx])); idx += 1
        T.append(int(tokens[idx])); idx += 1
        L.append(int(tokens[idx])); idx += 1
        g.append(float(tokens[idx])); idx += 1
        h.append(float(tokens[idx])); idx += 1

        for j in range(N):
            S[i][j] = int(tokens[idx]); idx += 1

    return N, E, T, L, g, h, S


def decode_sequence(perm, E, T, L, g, h, S):
    """
    HIZLANDIRILMIŞ DECODING (O(N^2) All-Pairs Separation):
    Tüm ikili ayırma kısıtlarını (m < k) koruyarak en düşük cezalı iniş zamanlarını hesaplar.
    """
    n = len(perm)
    x = [0.0] * n

    # 1. İleri Geçiş: O(N^2)
    for k in range(n):
        curr_p = perm[k]
        earliest = float(max(E[curr_p], T[curr_p]))
        for m in range(k):
            val = x[m] + S[perm[m]][curr_p]
            if val > earliest:
                earliest = val
        x[k] = earliest

    # 2. Ters Topolojik Geri Geçiş: O(N^2)
    for k in range(n - 1, -1, -1):
        p_k = perm[k]
        if x[k] > L[p_k]:
            x[k] = float(L[p_k])
        xk = x[k]
        for m in range(k):
            limit = xk - S[perm[m]][p_k]
            if x[m] > limit:
                x[m] = limit

    # 3. Uygunluk (Feasibility) Kontrolü: O(N^2)
    infeasible_penalty = 0.0
    for k in range(n):
        p = perm[k]
        xk = x[k]
        if xk < E[p] - 1e-6:
            infeasible_penalty += (E[p] - xk) * 10000.0
        elif xk > L[p] + 1e-6:
            infeasible_penalty += (xk - L[p]) * 10000.0
        for m in range(k):
            req = x[m] + S[perm[m]][p]
            if req > xk + 1e-6:
                infeasible_penalty += (req - xk) * 10000.0

    if infeasible_penalty > 0.0:
        return infeasible_penalty, x, False

    # 4. Hızlı Blok Kaydırma (Yalnızca gecikmeli uçak varsa çalışır)
    improved = True
    while improved:
        improved = False
        for k in range(n):
            if x[k] > T[perm[k]] + 1e-6:
                in_M = [False] * n
                in_M[k] = True
                M_list = [k]
                head = 0
                while head < len(M_list):
                    j = M_list[head]
                    head += 1
                    p_j = perm[j]
                    xj = x[j]
                    for i in range(j):
                        if not in_M[i] and abs((x[i] + S[perm[i]][p_j]) - xj) < 1e-6:
                            in_M[i] = True
                            M_list.append(i)

                slope = 0.0
                for u in M_list:
                    p_u = perm[u]
                    if x[u] > T[p_u] + 1e-6:
                        slope -= h[p_u]
                    else:
                        slope += g[p_u]

                if slope < -1e-6:
                    max_shift = float("inf")
                    for u in M_list:
                        p_u = perm[u]
                        xu = x[u]
                        diff_e = xu - E[p_u]
                        if diff_e < max_shift:
                            max_shift = diff_e
                        if xu > T[p_u] + 1e-6:
                            diff_t = xu - T[p_u]
                            if diff_t < max_shift:
                                max_shift = diff_t
                        for w in range(u):
                            if not in_M[w]:
                                diff_s = xu - (x[w] + S[perm[w]][p_u])
                                if diff_s < max_shift:
                                    max_shift = diff_s

                    if max_shift > 1e-6:
                        for u in M_list:
                            x[u] -= max_shift
                        improved = True
                        break

    # Toplam Ceza Maliyeti Hesabı
    total_cost = 0.0
    for k in range(n):
        p = perm[k]
        xk = x[k]
        if xk < T[p]:
            total_cost += g[p] * (T[p] - xk)
        elif xk > T[p]:
            total_cost += h[p] * (xk - T[p])

    return total_cost, x, True


# --- PENCERELİ KOMŞULUK OPERATÖRLERİ (Candidate Window = 12) ---

def get_tw_restricted_neighbors(perm, E, L, window=12):
    neighbors = []
    n = len(perm)
    for i in range(n):
        for j in range(i + 1, min(n, i + window + 1)):
            p1, p2 = perm[i], perm[j]
            if max(E[p1], E[p2]) <= min(L[p1], L[p2]):
                nb = perm[:]
                nb[i], nb[j] = nb[j], nb[i]
                neighbors.append(nb)
    return neighbors if neighbors else get_swap_neighbors(perm, window)


def get_swap_neighbors(perm, window=12):
    neighbors = []
    n = len(perm)
    for i in range(n):
        for j in range(i + 1, min(n, i + window + 1)):
            nb = perm[:]
            nb[i], nb[j] = nb[j], nb[i]
            neighbors.append(nb)
    return neighbors


def get_insertion_neighbors(perm, window=12):
    neighbors = []
    n = len(perm)
    for i in range(n):
        low = max(0, i - window)
        high = min(n, i + window + 1)
        for j in range(low, high):
            if i != j:
                nb = perm[:]
                plane = nb.pop(i)
                nb.insert(j, plane)
                neighbors.append(nb)
    return neighbors


# --- VARIABLE NEIGHBORHOOD SEARCH (VNS) ---

def local_search(perm, E, T, L, g, h, S, window=12, sample_size=80):
    """Hızlı örneklemeli First-Improvement yerel arama."""
    current = perm[:]
    best_cost, _, _ = decode_sequence(current, E, T, L, g, h, S)

    improved = True
    while improved:
        improved = False
        candidates = get_swap_neighbors(current, window) + get_insertion_neighbors(current, window)
        if len(candidates) > sample_size:
            candidates = random.sample(candidates, sample_size)
        else:
            random.shuffle(candidates)

        for cand in candidates:
            cost, _, _ = decode_sequence(cand, E, T, L, g, h, S)
            if cost < best_cost - 1e-6:
                best_cost = cost
                current = cand
                improved = True
                break
    return current, best_cost


def solve_alp_vns(N, E, T, L, g, h, S, max_iter=20, no_improve_limit=5, seed=42):
    random.seed(seed)
    start_time = time.time()
    window = 12

    current = sorted(range(N), key=lambda i: T[i])
    baseline_cost, _, _ = decode_sequence(current, E, T, L, g, h, S)

    current, current_cost = local_search(current, E, T, L, g, h, S, window)
    best_perm = current[:]
    best_cost = current_cost

    k_max = 3
    no_improve_count = 0

    for _ in range(max_iter):
        k = 1
        iter_improved = False
        while k <= k_max:
            if k == 1:
                neighborhood = get_tw_restricted_neighbors(current, E, L, window)
            elif k == 2:
                neighborhood = get_swap_neighbors(current, window)
            else:
                neighborhood = get_insertion_neighbors(current, window)

            shaken = random.choice(neighborhood)
            candidate, cand_cost = local_search(shaken, E, T, L, g, h, S, window)

            if cand_cost < current_cost - 1e-6:
                current = candidate
                current_cost = cand_cost
                k = 1
                if cand_cost < best_cost - 1e-6:
                    best_cost = cand_cost
                    best_perm = candidate[:]
                    iter_improved = True
            else:
                k += 1

        if iter_improved:
            no_improve_count = 0
        else:
            no_improve_count += 1
            if no_improve_count >= no_improve_limit:
                break

    elapsed = time.time() - start_time
    best_cost, best_x, is_feasible = decode_sequence(best_perm, E, T, L, g, h, S)
    return baseline_cost, best_cost, best_perm, best_x, is_feasible, elapsed


if __name__ == "__main__":
    instances = [f"airland{i}.txt" for i in range(1, 9)]
    bks_values = {
        "airland1.txt": 700,
        "airland2.txt": 1480,
        "airland3.txt": 820,
        "airland4.txt": 2520,
        "airland5.txt": 3100,
        "airland6.txt": 24442,
        "airland7.txt": 1550,
        "airland8.txt": 1950,
    }

    print(f"{'Instance':<14} {'N':<5} {'Baseline':<12} {'VNS Cost':<12} {'BKS':<8} {'Gap (%)':<10} {'Feasible':<10} {'Süre (sn)':<10}", flush=True)
    print("-" * 85, flush=True)

    for inst in instances:
        filepath = find_instance_file(inst)
        N, E, T, L, g, h, S = read_airland(filepath)
        base_cost, vns_cost, _, _, feasible, runtime = solve_alp_vns(
            N, E, T, L, g, h, S, max_iter=20, no_improve_limit=5, seed=42
        )
        bks = bks_values[inst]
        gap = ((vns_cost - bks) / bks) * 100.0
        print(
            f"{inst:<14} {N:<5} {base_cost:<12.2f} {vns_cost:<12.2f} "
            f"{bks:<8} {gap:<10.2f} {str(feasible):<10} {runtime:<10.4f}",
            flush=True
        )

Instance       N     Baseline     VNS Cost     BKS      Gap (%)    Feasible   Süre (sn) 
-------------------------------------------------------------------------------------
airland1.txt   10    700.00       700.00       700      0.00       True       0.2981    
airland2.txt   15    1500.00      1480.00      1480     0.00       True       1.3980    
airland3.txt   20    1730.00      820.00       820      0.00       True       2.2154    
airland4.txt   20    2520.00      2520.00      2520     0.00       True       4.0792    
airland5.txt   20    5420.00      3100.00      3100     0.00       True       2.9077    
airland6.txt   30    24442.00     24442.00     24442    0.00       True       0.5972    
airland7.txt   44    1550.00      1550.00      1550     0.00       True       2.5894    
airland8.txt   50    2480.00      2340.00      1950     20.00      True       16.7134   
